In [1]:
import glob
from pprint import pprint
import requests
import os
import mimetypes
from contextlib import ExitStack
from typing import List, Optional, Iterable
from settings import BASE_DIR
from pathlib import Path

from type import Transcript

URL = "http://localhost:8000/main/get-transcript"

def guess_mime(path: str) -> str:
    mime, _ = mimetypes.guess_type(path)
    return mime or "application/octet-stream"


def test_predict(
    image_paths: List[str],
    manga_name: str,
    chapter_name: str,
):
    """
    Gửi nhiều trang + ảnh nhân vật + danh sách tên nhân vật.

    Params:
      image_paths: danh sách file ảnh trang truyện.
      character_image_paths: danh sách file ảnh nhân vật (nếu None dùng DEFAULT_CHARACTER_IMAGES).
      character_names: nếu cung cấp sẽ dùng trực tiếp. Nếu None -> suy từ character_image_paths.
      auto_lower_character_names: nếu True sẽ chuyển tên về lowercase khi auto suy ra.
      url: endpoint API.
    """
    if not image_paths:
        raise ValueError("image_paths rỗng.")



    data_base = {
        "story_name": manga_name,
        "chapter_name": chapter_name,
    }

    with ExitStack() as stack:
        # Mở trang truyện
        chapter_file_objs = []
        for p in image_paths:
            f = stack.enter_context(Path(p).open("rb"))
            chapter_file_objs.append(
                ("chapter_pages", (os.path.basename(p), f, guess_mime(p)))
            )


        files = chapter_file_objs 

        # Chuyển dict data -> list tuple
        form_data = []
        for k, v in data_base.items():
            form_data.append((k, str(v)))

        timeout = 3000 # 
        resp = requests.post(URL, files=files, data=form_data, timeout=timeout)
        try:
            js = resp.json()
            pprint(js)
            for data in js:
                transcripts = [Transcript(**t) for t in data["transcript"]]
                transcripts_text = []
                for transcript in transcripts:
                    transcript = str(transcript)
                    transcripts_text.append(transcript)
                transcripts_text = "\n".join(transcripts_text)
                print("=" * 10)
                print(transcripts_text)
        except Exception as e:
            print(f"Error: {e}")
         


# Ví dụ chạy thử

print(1, 2)

1 2


In [6]:
CHAPTER = 134
PAGE = 6
test_predict(
    image_paths=[f"/home/aorus/workspaces/test_commic/data/Yule/Raw/{CHAPTER}/{PAGE}.jpg"],
    manga_name="Yule",
    chapter_name=str(CHAPTER),
)

[{'prose': 'Khu vườn tĩnh lặng bị phá vỡ bởi giọng nói giận dữ của Thái Thư '
           'Lệ. Cô đứng giữa những tán cây xanh mát và kiến trúc cổ kính, '
           'chiếc váy xanh lá cây nhạt dường như không đủ sức che giấu sự phẫn '
           'nộ đang bùng lên trong cô. Cô vung tay, chỉ về phía một bóng người '
           "xa xăm, giọng nói sắc bén vang vọng: “The Vuamo Scott say they're "
           "such a prestigious seat, but there's a huge disparity between the "
           'upper and lower tiers. The people at the top get everything they '
           'want…” Cô dừng lại, ánh mắt gằn guộc, rồi tiếp tục với một giọng '
           'đầy chua chát: “And for us at the lower tiers?”\n'
           '\n'
           'Ngay khi câu hỏi vừa dứt, một bàn tay bất ngờ vươn lên từ mặt đất, '
           'bao quanh là những đường năng lượng màu tím huyền bí. Bàn tay nắm '
           'chặt lấy một vật thể vô hình, nền trời phía sau chuyển sang màu '
           'tím sẫm, những đường kẻ năng lượng l

In [2]:
# chapter_names = [134, 135, 136, 137]
chapter_names = [134]
for chapter_name in chapter_names:
    folder_path = os.path.join(BASE_DIR, "test_commic/data/Yule/Raw", f"{chapter_name}/*.jpg")
    print(folder_path)
    image_paths = glob.glob(folder_path)
    image_paths.sort(key=lambda x: int(x.split("/")[-1].split(".")[0]))
    batch_size = 4
    total_batches = (len(image_paths) + batch_size - 1) // batch_size

    print(f"Tổng số trang: {len(image_paths)}, chia thành {total_batches} batch")

    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i+batch_size]
        current_batch = i // batch_size + 1
        print(f"\nĐang xử lý batch {current_batch}/{total_batches} ({len(batch)} trang)")
        
        test_predict(
            image_paths=batch,
            manga_name="Yule",
            chapter_name=f"{chapter_name}",  # Thêm số batch vào tên chương
        )

/home/aorus/workspaces/magiv2/test_commic/data/Yule/Raw/134/*.jpg
Tổng số trang: 23, chia thành 6 batch

Đang xử lý batch 1/6 (4 trang)
[{'prose': 'Khung tranh hiện ra, chia thành vô số ô nhỏ, mỗi ô là một mảnh '
           'ghép của thế giới này. Dư Lạc đứng chính giữa, khuôn mặt nghiêm '
           'nghị, mái tóc đen dựng đứng như thể đang đối mặt với một cơn bão. '
           'Hạ Kiều Nhu tự tin hiện diện ở phía dưới, bộ trang phục đỏ trắng '
           'lộng lẫy càng làm nổi bật vẻ kiêu hãnh của cô. Bên cạnh đó, Thái '
           'Thư Lệ khẽ nhếch mép, khói bốc lên từ chiếc cốc trong tay, ánh mắt '
           'sắc sảo. Cuối cùng, Tuyết Ly lạnh lùng nhìn thẳng về phía người '
           'xem, mái tóc tím đậm như một lời cảnh báo. Tất cả họ đều im lặng, '
           'chờ đợi điều gì đó sắp xảy ra.\n'
           '\n'
           'Không gian chuyển sang một căn phòng tối om. Liễu Nguyệt Nhi đứng '
           'bên trái, mái tóc vàng búi cao, nụ cười bí ẩn nở trên môi. Áo '
           'kh

Doraemon

In [19]:
chapter_name = 1
folder_path = os.path.join(BASE_DIR, "test_commic/data/doraemon/Raw", f"{chapter_name}/*.jpg")
print(folder_path)
image_paths = glob.glob(folder_path)
batch_size = 3
total_batches = (len(image_paths) + batch_size - 1) // batch_size

print(f"Tổng số trang: {len(image_paths)}, chia thành {total_batches} batch")

for i in range(0, len(image_paths), batch_size):
    batch = image_paths[i:i+batch_size]
    current_batch = i // batch_size + 1
    print(f"\nĐang xử lý batch {current_batch}/{total_batches} ({len(batch)} trang)")
    
    test_predict(
        image_paths=batch,
        manga_name="doraemon",
        chapter_name=f"{chapter_name}",  # Thêm số batch vào tên chương
    )

/home/aorus/workspaces/magiv2/test_commic/data/doraemon/Raw/1/*.jpg
Tổng số trang: 9, chia thành 3 batch

Đang xử lý batch 1/3 (3 trang)
Nobita speaking: 'MY FUTURE SELF?'
Nobita speaking: '"Yep, I just got home from school and..."'
Nobita speaking: 'FOUND MY HALF A YEAR SAVING IS MISSING!'
Nobita speaking: 'I USED THE TIME MACHINE TO COME BACK HERE EARLIER THAN YOU! NOW GIVE BACK MY MONEY!'
Nobita speaking: 'NO, THAT IS NOT NEEDED'
Nobita speaking: 'WE'RE THE SAME PERSON ANYWAY'
Nobita speaking: 'SOONER OR LATER, THIS MONEY BELONG TO ME, SO IT IS AS MUCH MINE AS IT IS YOURS'
Nobita speaking: 'NO WAY, I WORK REALLY HARD FOR IT. IT SHOULD BE MINE!'
Nobita speaking: 'I HAVE TO MASSAGE DAD'S BACK FOR THE LAST 6 MONTHS'
Nobita speaking: 'WE - TO EARN THAT MUCH MONEY!'
Nobita speaking: 'NO, IT'S MINE! BUT YOU AND ME ARE THE SAME PERSON, SO IT DOESN'T MATTER WHO'LL SPEND IT MONEY TO LEARN THAT MUCH MONEY'
Nobita speaking: 'NO, IT'S MINE!'
Nobita speaking: 'NO, THIS MONEY IS MINE!'
Doremon sp

In [10]:
image_paths = [os.path.join(BASE_DIR, "test_commic/data/doraemon/Raw", "1", "3.jpg")]
test_predict(
    image_paths=image_paths,
    manga_name="doraemon",
    chapter_name=str(1),
)


Other speaking: 'IS YOUR PAPER DONE?'
Nobita speaking: 'yes ne is'
Other speaking: 'I RECENTLY BOUGHT THESE.'
Other speaking: 'WOW! THESE ARE IMPORTED? THEY ARE TOP-CLASS!'
Other speaking: 'MUST BE COSTLY'
Other speaking: 'OF COURSE'
Other speaking: 'I WAS MOVING YOU GET ONE TOO'
Other speaking: 'I CAN'T AFFORD IT'
Other speaking: 'IS THAT YOUR CAR TOO?'
Other speaking: 'HOW CAN YOU AFFORD THEM? YOUR INCOME ISN'T ENOUGH!'
Other speaking: 'INSTALL MENT PAYING!'
Other speaking: 'LIKE IT'S The MODERN WORLD WAY'
Other speaking: 'IF YOU REALLY LIKE SOMETHING, YOU CAN JUST ACQUIRE IT NOW AND PAY LATER IN INSTALLMENT.'
Other speaking: 'FROM TIME TO TIME! EVERYONE IS DOING IT!'


[{'save_path': '/home/aorus/workspaces/magiv2/transcript_history/doraemon/1/3.json',
  'transcript': [{'speaker': 'Other',
    'target': 'Nobita',
    'text': 'IS YOUR\nPAPER\nDONE?',
    'text_speech_type': 'speaking'},
   {'speaker': 'Nobita',
    'target': 'Other',
    'text': 'yes\nne\nis',
    'text_speech_type': 'speaking'},
   {'speaker': 'Other',
    'target': 'Other',
    'text': 'I RECENTLY BOUGHT THESE.',
    'text_speech_type': 'speaking'},
   {'speaker': 'Other',
    'target': 'Other',
    'text': 'WOW! THESE ARE\nIMPORTED? THEY\nARE TOP-CLASS!',
    'text_speech_type': 'speaking'},
   {'speaker': 'Other',
    'target': 'Other',
    'text': 'MUST BE\nCOSTLY',
    'text_speech_type': 'speaking'},
   {'speaker': 'Other',
    'target': 'Other',
    'text': 'OF COURSE',
    'text_speech_type': 'speaking'},
   {'speaker': 'Other',
    'target': 'Other',
    'text': 'I WAS\nMOVING YOU\nGET ONE\nTOO',
    'text_speech_type': 'speaking'},
   {'speaker': 'Other',
    'target': 'Oth

In [33]:
chapter_names = [84, 85, 86]
for chapter_name in chapter_names:
    folder_path = os.path.join(BASE_DIR, "test_commic/data/tien_boi/Raw", f"{chapter_name}/*.jpg")
    print(folder_path)
    image_paths = glob.glob(folder_path)
    image_paths.sort(key=lambda x: int(x.split("/")[-1].split(".")[0]))
    batch_size = 2
    total_batches = (len(image_paths) + batch_size - 1) // batch_size

    print(f"Tổng số trang: {len(image_paths)}, chia thành {total_batches} batch")

    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i+batch_size]
        current_batch = i // batch_size + 1
        print(f"\nĐang xử lý batch {current_batch}/{total_batches} ({len(batch)} trang)")
        
        test_predict(
            image_paths=batch,
            manga_name="tien_boi",
            chapter_name=f"{chapter_name}",  # Thêm số batch vào tên chương
        )

/home/aorus/workspaces/magiv2/test_commic/data/tien_boi/Raw/84/*.jpg
Tổng số trang: 14, chia thành 7 batch

Đang xử lý batch 1/7 (2 trang)
[{'prose': 'Đường phố nhộn nhịp với những chiếc đèn lồng treo cao, một cửa '
           'hàng có mái che nhỏ nhắn nằm nép mình bên cạnh. Bỗng nhiên, một '
           'người đàn ông dừng bước, ngước nhìn lên với vẻ mặt ngạc nhiên. '
           'Người bạn đồng hành bên cạnh, một người phụ nữ, cũng dõi theo anh, '
           'trong khi người còn lại, đứng bên phải, hướng ánh mắt về phía cô. '
           'Cả ba dường như đều bị một điều gì đó thu hút sự chú ý.\n'
           '\n'
           'Vài phút sau, màn hình điện thoại hiển thị 17:40. Một biểu tượng '
           'hình tròn màu xanh lam và một hình tam giác màu đỏ nhấp nháy trên '
           'màn hình, như báo hiệu một điều gì đó sắp xảy ra.\n'
           '\n'
           'Trong một căn phòng chờ trang trọng, một người phụ nữ tóc trắng '
           'ngồi thẳng lưng, ánh mắt nhìn về phía trước với vẻ 

In [8]:
image_paths = [os.path.join(BASE_DIR, "test_commic/data/tien_boi/Raw", "82", "10.jpg")]
test_predict(
    image_paths=image_paths,
    manga_name="tien_boi",
    chapter_name=str(82),
)


[{'prose': 'Đêm pháo hoa rực sáng bầu trời, những tia sáng đủ màu vẽ nên một '
           'bức tranh ngoạn mục giữa lòng thành phố. Hai người đứng cạnh nhau, '
           'ngước nhìn lên, hoàn toàn bị mê hoặc bởi màn trình diễn ánh sáng. '
           'Cô gái, với mái tóc ngắn sáng màu, bỗng mở to mắt, há họng, như '
           'thể bị đánh thức bởi một điều gì đó. Khuôn mặt cô thoáng hiện vẻ '
           'ngạc nhiên, pha lẫn một chút bối rối.\n'
           '\n'
           'Một khoảnh khắc tĩnh lặng bao trùm lấy họ. Người phụ nữ, với mái '
           'tóc sáng màu và ánh mắt hờ hững, nhắm mắt lại, như đang cố gắng '
           'kìm nén một điều gì đó. Người đàn ông bên cạnh, với mái tóc đen, '
           'chăm chú nhìn cô, ánh mắt đầy vẻ quan tâm.\n'
           '\n'
           'Rồi đột ngột, cô gái lại ngước nhìn lên bầu trời, tay giơ cao như '
           'muốn chạm vào những bông hoa nở rộ. Vẫn còn vẻ ngạc nhiên trên '
           'khuôn mặt, cô dường như đang cố gắng tìm kiếm điều gì đ